In [2]:
import os
import warnings
import joblib
import pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

warnings.filterwarnings("ignore", category=DeprecationWarning)

# Initialize FastAPI App
app = FastAPI(
    title="E-Commerce AI Engine API",
    description="REST API for Churn Prediction & Multi-Tiered Product Recommendations",
    version="1.0.0",
)

# --- Load Models ---
MODELS_DIR = r"D:\ecomai\models"

# 1. Churn Model
churn_model = joblib.load(
    os.path.join(MODELS_DIR, "gradient_boosting_tuned.pkl")
)

# 2. Recommendation Engines
engine1_sim = joblib.load(
    os.path.join(MODELS_DIR, "engine1_item_similarity.pkl")
)
engine2_rules = joblib.load(os.path.join(MODELS_DIR, "engine2_cart_rules.pkl"))
engine3_svd = joblib.load(
    os.path.join(MODELS_DIR, "engine3_personalized_svd.pkl")
)


# --- Pydantic Data Schemas ---
class ChurnInput(BaseModel):
  Recency: float
  Frequency: float
  Monetary: float
  AvgOrderValue: float
  TenureDays: float


class ItemRecInput(BaseModel):
  product_name: str
  top_n: int = 5


class UserRecInput(BaseModel):
  customer_id: int
  top_n: int = 5


# --- API Endpoints ---


@app.get("/")
def health_check():
  return {
      "status": "online",
      "message": "E-Commerce AI REST API is running smoothly.",
  }


# Endpoint 1: Churn Prediction
@app.post("/predict/churn")
def predict_churn(data: ChurnInput):
  input_df = pd.DataFrame([data.model_dump()])

  if hasattr(churn_model, "predict_proba"):
    prob = float(churn_model.predict_proba(input_df)[0][1])
    prediction = int(prob >= 0.5)
  else:
    prediction = int(churn_model.predict(input_df)[0])
    prob = float(prediction)

  risk_level = (
      "High" if prob >= 0.7 else ("Medium" if prob >= 0.3 else "Low")
  )

  return {
      "churn_prediction": prediction,
      "churn_probability": round(prob, 4),
      "risk_level": risk_level,
  }


# Endpoint 2: Product Page Similar Items
@app.post("/recommend/similar-items")
def recommend_similar(data: ItemRecInput):
  if data.product_name not in engine1_sim.columns:
    raise HTTPException(
        status_code=404,
        detail=f"Product '{data.product_name}' not found in catalog.",
    )

  scores = (
      engine1_sim[data.product_name]
      .sort_values(ascending=False)
      .iloc[1 : data.top_n + 1]
  )
  recommendations = [
      {"product_name": prod, "similarity_score": round(score, 4)}
      for prod, score in scores.items()
  ]

  return {"product_name": data.product_name, "recommendations": recommendations}


# Endpoint 3: Cart Cross-Sell Bundles
@app.post("/recommend/cart-bundles")
def recommend_cart(data: ItemRecInput):
  matching = engine2_rules[
      engine2_rules["antecedents"].apply(lambda x: data.product_name in x)
  ]

  if matching.empty:
    return {
        "product_name": data.product_name,
        "message": "No cross-sell rules found for this item.",
        "bundles": [],
    }

  top_rules = matching.head(data.top_n)
  results = []
  for _, row in top_rules.iterrows():
    results.append({
        "recommended_items": row["consequents"],
        "confidence": round(row["confidence"], 4),
        "lift": round(row["lift"], 4),
    })

  return {"product_name": data.product_name, "bundles": results}


# Endpoint 4: Personalized User Feed
@app.post("/recommend/user-feed")
def recommend_user(data: UserRecInput):
  matrix = engine3_svd["user_item_matrix"]

  if data.customer_id not in matrix.index:
    raise HTTPException(
        status_code=404,
        detail=f"Customer ID {data.customer_id} not found.",
    )

  u_idx = matrix.index.get_loc(data.customer_id)
  predicted_ratings = (
      engine3_svd["user_factors"][u_idx] @ engine3_svd["item_factors"]
  )
  preds_series = pd.Series(predicted_ratings, index=matrix.columns)

  already_bought = matrix.loc[data.customer_id]
  preds_series = preds_series[already_bought == 0]
  top_preds = preds_series.sort_values(ascending=False).head(data.top_n)

  recs = [
      {"product_name": prod, "predicted_score": round(score, 4)}
      for prod, score in top_preds.items()
  ]

  return {"customer_id": data.customer_id, "recommendations": recs}

ModuleNotFoundError: No module named 'fastapi'